# 교차 평가 (Cross-evaluation)

이미 끝난 AI의 본시험 기록(`paired_test.json`)을 **다시 학습하지 않고** 27개 점수표 모두로 다시 채점합니다.

- 런타임: **CPU, 고용량 RAM** (GPU 불필요: 시간 대부분이 Drive 파일 읽기와 JSON 해석)
- 순서대로 실행하세요. 1번 셀의 Drive 허용은 직접 눌러야 합니다.
- 한 번 뽑은 AI는 캐시되므로, 다시 실행하면 새로 끝난 AI만 읽습니다.

In [ ]:
# 1. Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 코드 받기
REPO_COMMIT = 'main'
!rm -rf /content/foodcourt-cross-eval
!git clone -q https://github.com/phj9187/foodcourt-cross-eval /content/foodcourt-cross-eval
!cd /content/foodcourt-cross-eval && git checkout -q $REPO_COMMIT && git log --oneline -1
!pip -q install orjson
import os, sys
os.chdir('/content/foodcourt-cross-eval'); sys.path.insert(0, '/content/foodcourt-cross-eval')
print('CPU cores:', os.cpu_count())

In [ ]:
# 3. 대상 실험 (끝난 AI만 읽음)
BASE = '/content/drive/MyDrive/foodcourt_adaptive_sim'
EXPERIMENTS = ['extended_main_20261108_001', 'extended_taste_20261110_001']
OUT = f'{BASE}/cross_eval'
WORKERS = 0   # 0 = 코어 수와 메모리로 자동 결정
for e in EXPERIMENTS:
    print(e, os.path.isdir(f'{BASE}/{e}'))

In [ ]:
# 4. 추출: paired_test.json -> 작은 .npz (병렬, 캐시)
for e in EXPERIMENTS:
    print('=====', e)
    !python -m cross_eval.extract "{BASE}/{e}" "{OUT}/{e}/cache" --workers {WORKERS}

In [ ]:
# 5. 재채점과 요약
for e in EXPERIMENTS:
    print('=====', e)
    !python -m cross_eval.analyze "{OUT}/{e}/cache" "{OUT}/{e}/results"


In [ ]:
# 6. 결과 보기
import pandas as pd
from IPython.display import display, Image
for e in EXPERIMENTS:
    r = f'{OUT}/{e}/results'
    print('=====', e)
    display(pd.read_csv(f'{r}/self_vs_best_stable.csv'))
    display(pd.read_csv(f'{r}/generalist_stable.csv').head(10))
    display(pd.read_csv(f'{r}/common_metrics_stable.csv').sort_values('d_profit_won_per_court_day', ascending=False))
    display(Image(f'{r}/heatmap_stable.png'))